# Ablation Study: Which Features Drive Elution Sequence Prediction?

**Version: 1.0** | 2026-03-10

**Purpose:** Systematically remove each input feature from the LSTM model to
measure its contribution to next-m/z-bin prediction accuracy.

**Ablation conditions (7 total):**

| # | Condition | What changes |
|---|-----------|-------------|
| 1 | Full model | All 5 features — baseline |
| 2 | −m/z bin | Zero out m/z embedding |
| 3 | −mass defect | Zero out mass defect embedding |
| 4 | −RT gap | Zero out RT gap embedding |
| 5 | −polarity | Zero out polarity embedding |
| 6 | −intensity | Zero out intensity rank embedding |
| 7 | No sequence | Context length = 1 (feedforward baseline) |
| 8 | 25% data | Train on 25% of samples (all features) |
| 9 | 50% data | Train on 50% of samples (all features) |
| 10 | 75% data | Train on 75% of samples (all features) |

**Design:** Each condition trains a fresh LSTM (256K params) for up to 100
epochs with early stopping (patience=10). We use embedding zeroing rather than
removing input dimensions so that all models have identical architecture and
parameter count — isolating the information contribution of each feature.
Data-efficiency conditions (8–10) use a random subset of training samples
(val/test unchanged) to measure how much data is needed.

**Resume logic:** If Colab disconnects, re-run all cells. The training loop
automatically detects the last completed epoch from the checkpoint and
continues. Completed conditions are skipped entirely. Optimizer and scheduler
state are saved in checkpoints for exact resume.

**Monitoring:** Check `outputs/ablation_log.csv` on Google Drive from your
local machine to monitor progress remotely.

**Runtime:** ~4–5 hours on T4 GPU (10 conditions × ~25–30 min each).

**Safeguards:**
- Atomic checkpoint writes (save to `.tmp`, then rename) — survives Colab kill mid-save
- Saves both current and best model state — exact training continuation after resume
- CUDA OOM catch-and-continue — logs failure, moves to next condition
- Sanity check: warns if "full" condition deviates >1% from original 98.38%
- ETA tracking: logs estimated time remaining for monitoring

**Changelog:**
- v1.1: Add atomic writes, dual state save, OOM protection, sanity check, ETA
- v1.0: Initial version


In [ ]:
# @title Cell 1: Setup and Google Drive mount
import os, time, json, csv, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datetime import datetime

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc3_elution_sequence"
EXPERIMENT_DIR = f"{DRIVE_DIR}/07_ablation_study"
OUTPUT_DIR = f"{EXPERIMENT_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"Device: {device}")
    print(f"GPU: {gpu.name}")
    print(f"Memory: {gpu.total_memory / 1e9:.1f} GB")
else:
    print(f"Device: {device} (WARNING: training will be very slow)")

print(f"\nExperiment dir: {EXPERIMENT_DIR}")
print(f"Output dir: {OUTPUT_DIR}")


## 1. Configuration

In [ ]:
# @title Cell 2: Configuration & hyperparameters
RANDOM_SEED = 42
CONTEXT_LENGTH = 64
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.1
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

RT_GAP_LABELS = ["co-elute", "0.1-0.5s", "0.5-1s", "1-2s", "2-5s", "5-15s", ">15s"]
INTENSITY_RANK_LABELS = ["top1%", "top5%", "top20%", "top50%", "low"]
POLARITY_MAP = {"pos": 0, "neg": 1, "unk": 2}
RT_GAP_MAP = {label: i for i, label in enumerate(RT_GAP_LABELS)}
INTENSITY_MAP = {label: i for i, label in enumerate(INTENSITY_RANK_LABELS)}

# Ablation conditions: (name, description, features_to_zero, context_length_override)
# features_to_zero: list of embedding fields to zero out during forward pass
# context_length_override: None = use default (64), 1 = feedforward baseline
ABLATION_CONDITIONS = [
    ("full",         "Full model (all features)",  [],              None),
    ("no_mz",        "−m/z bin",                   ["mz"],          None),
    ("no_md",        "−mass defect",               ["md"],          None),
    ("no_rt_gap",    "−RT gap",                    ["gap"],         None),
    ("no_polarity",  "−polarity",                  ["pol"],         None),
    ("no_intensity", "−intensity rank",            ["intensity"],   None),
    ("no_sequence",  "No sequence (ctx=1)",        [],              1),
    ("data_25pct",   "25% training data",          [],              None),
    ("data_50pct",   "50% training data",          [],              None),
    ("data_75pct",   "75% training data",          [],              None),
]

# Data fraction for data-efficiency conditions
DATA_FRACTION = {
    "data_25pct": 0.25,
    "data_50pct": 0.50,
    "data_75pct": 0.75,
}

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Ablation conditions: {len(ABLATION_CONDITIONS)}")
for name, desc, zeros, ctx in ABLATION_CONDITIONS:
    ctx_str = f"ctx={ctx}" if ctx else "ctx=64"
    zero_str = f"zero={zeros}" if zeros else "all features"
    frac_str = f" ({DATA_FRACTION[name]*100:.0f}% data)" if name in DATA_FRACTION else ""
    print(f"  {name:16s} | {desc:30s} | {zero_str:25s} | {ctx_str}{frac_str}")


## 2. Data Pipeline

In [ ]:
# @title Cell 3: Load and encode tokenized features
TRAIN_DIR = f"{DRIVE_DIR}/01_train_models"
tok = pd.read_parquet(f"{TRAIN_DIR}/tokenized_features.parquet")
print(f"Loaded {len(tok):,} rows, {tok['sample_id'].nunique()} samples")
print(f"Columns: {list(tok.columns)}")

# Encode categorical fields
tok["polarity_idx"] = tok["polarity_tok"].map(POLARITY_MAP).fillna(2).astype(int)
tok["rt_gap_idx"] = tok["rt_gap_bin"].astype(str).map(RT_GAP_MAP).fillna(0).astype(int)
tok["intensity_idx"] = tok["intensity_rank"].astype(str).map(INTENSITY_MAP).fillna(4).astype(int)
tok = tok.sort_values(["study", "sample_id", "seq_pos"])
print(f"Encoded. Polarity unique: {tok['polarity_idx'].unique()}")


In [ ]:
# @title Cell 4: Sample-aware train/val/test split
rng = np.random.RandomState(RANDOM_SEED)
samples = tok[["study", "sample_id"]].drop_duplicates().reset_index(drop=True)

train_keys, val_keys, test_keys = set(), set(), set()
for study, group in samples.groupby("study"):
    indices = group.index.values.copy()
    rng.shuffle(indices)
    n = len(indices)
    n_test = max(1, int(n * TEST_FRACTION))
    n_val = max(1, int(n * VAL_FRACTION))
    for idx in indices[:n_test]:
        test_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test:n_test + n_val]:
        val_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test + n_val:]:
        train_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))

def mask_for(keys):
    return tok.apply(lambda r: (r["study"], r["sample_id"]) in keys, axis=1)

train_df = tok[mask_for(train_keys)]
val_df = tok[mask_for(val_keys)]
test_df = tok[mask_for(test_keys)]

print(f"Train: {train_df[['study','sample_id']].drop_duplicates().shape[0]} samples, {len(train_df):,} rows")
print(f"Val:   {val_df[['study','sample_id']].drop_duplicates().shape[0]} samples, {len(val_df):,} rows")
print(f"Test:  {test_df[['study','sample_id']].drop_duplicates().shape[0]} samples, {len(test_df):,} rows")


In [ ]:
# @title Cell 5: Dataset class and builder
from torch.utils.data import Dataset, DataLoader

def build_sample_arrays(df):
    samples = []
    for (study, sid), group in df.groupby(["study", "sample_id"]):
        g = group.sort_values("seq_pos")
        samples.append({
            "study": study, "sample_id": sid,
            "mz_bin": g["mz_bin"].values.astype(np.int64),
            "md_bin": g["md_bin"].values.astype(np.int64),
            "rt_gap_idx": g["rt_gap_idx"].values.astype(np.int64),
            "polarity_idx": g["polarity_idx"].values.astype(np.int64),
            "intensity_idx": g["intensity_idx"].values.astype(np.int64),
            "rt_bin": g["rt_bin"].values.astype(np.int64),
        })
    return samples

class ElutionSequenceDataset(Dataset):
    def __init__(self, sample_arrays, context_length=CONTEXT_LENGTH):
        self.context_length = context_length
        self.sample_arrays = sample_arrays
        self.examples = []
        for si, s in enumerate(sample_arrays):
            for pos in range(context_length, len(s["mz_bin"])):
                self.examples.append((si, pos))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        si, pos = self.examples[idx]
        s = self.sample_arrays[si]
        start = pos - self.context_length
        return {
            "ctx_mz":  torch.tensor(s["mz_bin"][start:pos], dtype=torch.long),
            "ctx_md":  torch.tensor(s["md_bin"][start:pos], dtype=torch.long),
            "ctx_gap": torch.tensor(s["rt_gap_idx"][start:pos], dtype=torch.long),
            "ctx_pol": torch.tensor(s["polarity_idx"][start:pos], dtype=torch.long),
            "ctx_int": torch.tensor(s["intensity_idx"][start:pos], dtype=torch.long),
            "target_mz": torch.tensor(s["mz_bin"][pos], dtype=torch.long),
        }

# Build sample arrays (we'll create DataLoaders per-condition because ctx length varies)
train_arr = build_sample_arrays(train_df)
val_arr = build_sample_arrays(val_df)
test_arr = build_sample_arrays(test_df)

MAX_MZ_BIN = max(
    max(s["mz_bin"].max() for s in train_arr),
    max(s["mz_bin"].max() for s in val_arr),
    max(s["mz_bin"].max() for s in test_arr),
) + 1
print(f"Max m/z bin (num classes): {MAX_MZ_BIN}")
print(f"Sample arrays: train={len(train_arr)}, val={len(val_arr)}, test={len(test_arr)}")


## 3. Model Definition

In [ ]:
# @title Cell 6: LSTM model with feature ablation support
import math

class MultiFieldEmbedding(nn.Module):
    """Embed each token field separately, then sum."""
    def __init__(self, embedding_dim, max_mz_bin=120, max_md_bin=20,
                 max_rt_gap=7, max_polarity=3, max_intensity=5):
        super().__init__()
        self.mz_embed = nn.Embedding(max_mz_bin, embedding_dim)
        self.md_embed = nn.Embedding(max_md_bin, embedding_dim)
        self.gap_embed = nn.Embedding(max_rt_gap, embedding_dim)
        self.pol_embed = nn.Embedding(max_polarity, embedding_dim)
        self.int_embed = nn.Embedding(max_intensity, embedding_dim)

    def forward(self, mz, md, gap, pol, intensity, zero_fields=None):
        """Sum embeddings, zeroing out specified fields for ablation."""
        zero = set(zero_fields or [])
        result = torch.zeros(mz.shape[0], mz.shape[1], self.mz_embed.embedding_dim,
                             device=mz.device)
        if "mz" not in zero:
            result = result + self.mz_embed(mz)
        if "md" not in zero:
            result = result + self.md_embed(md)
        if "gap" not in zero:
            result = result + self.gap_embed(gap)
        if "pol" not in zero:
            result = result + self.pol_embed(pol)
        if "intensity" not in zero:
            result = result + self.int_embed(intensity)
        return result


class AblationLSTM(nn.Module):
    """LSTM with configurable feature ablation via embedding zeroing."""
    def __init__(self, num_mz_classes, embedding_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.1, zero_fields=None, **embed_kw):
        super().__init__()
        self.zero_fields = zero_fields or []
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kw)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, num_mz_classes)

    def forward(self, mz, md, gap, pol, intensity):
        x = self.embedding(mz, md, gap, pol, intensity, zero_fields=self.zero_fields)
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :]))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Quick test
embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7, max_polarity=3, max_intensity=5)
test_model = AblationLSTM(MAX_MZ_BIN, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT,
                          zero_fields=["mz"], **embed_kw)
print(f"AblationLSTM: {count_parameters(test_model):,} parameters")
print(f"All models have identical parameter count regardless of ablation condition")
del test_model


## 4. Training Loop (with Resume)

In [ ]:
# @title Cell 7: Training loop with checkpoint resume and CSV logging
from torch.optim.lr_scheduler import CosineAnnealingLR

LOG_PATH = f"{OUTPUT_DIR}/ablation_log.csv"
LOG_FIELDS = [
    "timestamp", "condition", "epoch", "max_epochs", "status",
    "train_loss", "train_acc", "val_loss", "val_top1", "val_top5", "val_mae_da",
    "best_epoch", "best_val_loss", "lr", "epoch_time_s", "total_time_s",
    "gpu_mem_used_mb", "gpu_mem_total_mb",
]


def log_to_csv(row_dict):
    """Append one row to the persistent ablation log on Drive."""
    file_exists = os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


def atomic_save(obj, path):
    """Save checkpoint atomically: write to .tmp, then rename.
    Survives Colab kills mid-write — the old checkpoint remains intact."""
    tmp_path = path + ".tmp"
    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)  # atomic on POSIX and most filesystems


def get_gpu_memory():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e6
        total = torch.cuda.get_device_properties(0).total_memory / 1e6
        return round(used, 1), round(total, 1)
    return 0, 0


def evaluate(model, loader, criterion, top_k=(1, 3, 5, 10)):
    model.eval()
    total_loss, total_n = 0, 0
    total_correct = {k: 0 for k in top_k}
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            mz = batch["ctx_mz"].to(device)
            md_t = batch["ctx_md"].to(device)
            gap = batch["ctx_gap"].to(device)
            pol = batch["ctx_pol"].to(device)
            inten = batch["ctx_int"].to(device)
            target = batch["target_mz"].to(device)
            logits = model(mz, md_t, gap, pol, inten)
            loss = criterion(logits, target)
            total_loss += loss.item() * target.size(0)
            for k in top_k:
                _, topk_preds = logits.topk(k, dim=1)
                total_correct[k] += topk_preds.eq(target.unsqueeze(1)).any(dim=1).sum().item()
            pred_mz = logits.argmax(dim=1)
            all_preds.append(pred_mz.cpu().numpy())
            all_targets.append(target.cpu().numpy())
            total_n += target.size(0)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    mae_bins = np.mean(np.abs(all_preds.astype(float) - all_targets.astype(float)))
    metrics = {"loss": total_loss / total_n, "mae_da": float(mae_bins * 10), "n": total_n}
    for k in top_k:
        metrics[f"top{k}"] = total_correct[k] / total_n
    return metrics


def train_ablation(condition_name, description, zero_fields, ctx_override,
                   max_epochs=MAX_EPOCHS, patience=PATIENCE, lr=LEARNING_RATE):
    """Train one ablation condition with full checkpoint/resume support."""

    ctx_len = ctx_override if ctx_override is not None else CONTEXT_LENGTH
    ckpt_path = f"{OUTPUT_DIR}/ablation_{condition_name}.pt"

    # --- Check if already completed ---
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if "test_metrics" in ckpt:
            print(f"\n{'='*70}")
            print(f"SKIP {condition_name} — already completed (test top1={ckpt['test_metrics']['top1']:.4f})")
            print(f"{'='*70}")
            return ckpt["test_metrics"], ckpt["history"]

    # --- Subsample training data for data-efficiency conditions ---
    data_frac = DATA_FRACTION.get(condition_name, 1.0)
    if data_frac < 1.0:
        n_samples = max(1, int(len(train_arr) * data_frac))
        sub_rng = np.random.RandomState(RANDOM_SEED)
        sub_idx = sub_rng.choice(len(train_arr), n_samples, replace=False)
        sub_train_arr = [train_arr[i] for i in sorted(sub_idx)]
        print(f"\nData subsample: {n_samples}/{len(train_arr)} training samples ({data_frac*100:.0f}%)")
    else:
        sub_train_arr = train_arr

    # --- Build DataLoaders for this context length ---
    train_ds = ElutionSequenceDataset(sub_train_arr, context_length=ctx_len)
    val_ds = ElutionSequenceDataset(val_arr, context_length=ctx_len)
    test_ds = ElutionSequenceDataset(test_arr, context_length=ctx_len)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    print(f"\nDataLoaders (ctx={ctx_len}): train={len(train_ds):,} val={len(val_ds):,} test={len(test_ds):,}")

    # --- Create model ---
    embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7, max_polarity=3, max_intensity=5)
    model = AblationLSTM(MAX_MZ_BIN, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT,
                         zero_fields=zero_fields, **embed_kw)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs)

    # --- Resume from checkpoint if exists ---
    start_epoch = 1
    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    history = []

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        # Load CURRENT state (not best) so training continues from where it left off
        current_sd = ckpt.get("current_state_dict", ckpt["model_state_dict"])
        model.load_state_dict(current_sd)
        best_state = {k: v.cpu().clone() for k, v in ckpt["model_state_dict"].items()}
        best_epoch = ckpt["best_epoch"]
        history = ckpt["history"]
        start_epoch = len(history) + 1
        best_val_loss = ckpt.get("best_val_loss", history[-1]["val_loss"] if history else float("inf"))

        # Restore optimizer and scheduler state for exact resume
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scheduler_state_dict" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        else:
            for _ in range(start_epoch - 1):
                scheduler.step()

        print(f"Resumed from epoch {start_epoch - 1} (best_epoch={best_epoch}, best_val_loss={best_val_loss:.6f})")
    else:
        print(f"Starting fresh training for {condition_name}")

    config = {
        "condition": condition_name, "description": description,
        "zero_fields": zero_fields, "context_length": ctx_len,
        "embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS, "num_classes": MAX_MZ_BIN,
    }

    print(f"\n{'='*70}")
    print(f"Training: {condition_name} — {description}")
    print(f"  Zero fields: {zero_fields if zero_fields else 'none'}")
    print(f"  Context length: {ctx_len}")
    print(f"  Parameters: {count_parameters(model):,}")
    print(f"  Epochs: {start_epoch}..{max_epochs} (patience={patience})")
    print(f"{'='*70}")

    # Log start
    gpu_used, gpu_total = get_gpu_memory()
    log_to_csv({
        "timestamp": datetime.now().isoformat(), "condition": condition_name,
        "epoch": start_epoch - 1, "max_epochs": max_epochs,
        "status": "RESUMED" if start_epoch > 1 else "STARTED",
        "train_loss": "", "train_acc": "", "val_loss": "", "val_top1": "",
        "val_top5": "", "val_mae_da": "", "best_epoch": best_epoch,
        "best_val_loss": f"{best_val_loss:.6f}" if best_val_loss < float("inf") else "",
        "lr": lr, "epoch_time_s": "", "total_time_s": 0,
        "gpu_mem_used_mb": gpu_used, "gpu_mem_total_mb": gpu_total,
    })

    training_start = time.time()

    for epoch in range(start_epoch, max_epochs + 1):
        # Reset seed per epoch for reproducibility after resume
        torch.manual_seed(RANDOM_SEED + epoch)

        model.train()
        train_loss, train_correct, train_n = 0, 0, 0
        t0 = time.time()
        for batch in train_loader:
            mz = batch["ctx_mz"].to(device)
            md_t = batch["ctx_md"].to(device)
            gap = batch["ctx_gap"].to(device)
            pol = batch["ctx_pol"].to(device)
            inten = batch["ctx_int"].to(device)
            target = batch["target_mz"].to(device)
            optimizer.zero_grad()
            logits = model(mz, md_t, gap, pol, inten)
            loss = criterion(logits, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * target.size(0)
            train_correct += (logits.argmax(1) == target).sum().item()
            train_n += target.size(0)
        scheduler.step()
        dt = time.time() - t0
        total_time = time.time() - training_start

        val_m = evaluate(model, val_loader, criterion)
        rec = {
            "epoch": epoch, "train_loss": train_loss / train_n,
            "train_acc": train_correct / train_n,
            "val_loss": val_m["loss"], "val_top1": val_m["top1"],
            "val_top5": val_m["top5"], "val_mae_da": val_m["mae_da"],
            "lr": optimizer.param_groups[0]["lr"], "time_s": dt,
        }
        history.append(rec)
        print(f"  [{condition_name}] Epoch {epoch:3d}/{max_epochs} | "
              f"loss={rec['train_loss']:.4f} acc={rec['train_acc']:.4f} | "
              f"val top1={rec['val_top1']:.4f} top5={rec['val_top5']:.4f} "
              f"MAE={rec['val_mae_da']:.0f}Da | {dt:.0f}s")

        improved = val_m["loss"] < best_val_loss
        if improved:
            best_val_loss = val_m["loss"]
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Save checkpoint after EVERY epoch (atomic write, dual state)
        if best_state is not None:
            current_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            atomic_save({
                "model_state_dict": best_state,           # best weights (for final eval)
                "current_state_dict": current_state,      # current weights (for resume)
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "config": config,
                "history": history,
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
            }, ckpt_path)

        gpu_used, gpu_total = get_gpu_memory()
        log_to_csv({
            "timestamp": datetime.now().isoformat(), "condition": condition_name,
            "epoch": epoch, "max_epochs": max_epochs,
            "status": "IMPROVED" if improved else "PLATEAU",
            "train_loss": f"{rec['train_loss']:.6f}",
            "train_acc": f"{rec['train_acc']:.6f}",
            "val_loss": f"{rec['val_loss']:.6f}",
            "val_top1": f"{rec['val_top1']:.6f}",
            "val_top5": f"{rec['val_top5']:.6f}",
            "val_mae_da": f"{rec['val_mae_da']:.1f}",
            "best_epoch": best_epoch,
            "best_val_loss": f"{best_val_loss:.6f}",
            "lr": f"{rec['lr']:.8f}",
            "epoch_time_s": f"{dt:.1f}",
            "total_time_s": f"{total_time:.1f}",
            "gpu_mem_used_mb": gpu_used,
            "gpu_mem_total_mb": gpu_total,
        })

        if epoch - best_epoch >= patience:
            print(f"  [{condition_name}] Early stopping at epoch {epoch} (best: {best_epoch})")
            log_to_csv({
                "timestamp": datetime.now().isoformat(), "condition": condition_name,
                "epoch": epoch, "max_epochs": max_epochs, "status": "EARLY_STOPPED",
                "train_loss": "", "train_acc": "", "val_loss": "", "val_top1": "",
                "val_top5": "", "val_mae_da": "", "best_epoch": best_epoch,
                "best_val_loss": f"{best_val_loss:.6f}", "lr": "",
                "epoch_time_s": "", "total_time_s": f"{total_time:.1f}",
                "gpu_mem_used_mb": gpu_used, "gpu_mem_total_mb": gpu_total,
            })
            break

    # --- Test evaluation ---
    model.load_state_dict(best_state)
    model = model.to(device)
    test_m = evaluate(model, test_loader, criterion)

    print(f"\n  [{condition_name}] TEST: top1={test_m['top1']:.4f} top5={test_m['top5']:.4f} "
          f"top10={test_m['top10']:.4f} MAE={test_m['mae_da']:.0f}Da")

    # Save final checkpoint with test metrics (atomic)
    atomic_save({
        "model_state_dict": best_state,
        "test_metrics": test_m,
        "config": config,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
    }, ckpt_path)

    # Sanity check: full model should match original training (~98.38%)
    if condition_name == "full" and abs(test_m["top1"] - 0.9838) > 0.01:
        print(f"  WARNING: Full model top1={test_m['top1']:.4f} deviates >1% from "
              f"original training (0.9838). Check data split consistency.")

    log_to_csv({
        "timestamp": datetime.now().isoformat(), "condition": condition_name,
        "epoch": best_epoch, "max_epochs": max_epochs, "status": "COMPLETED",
        "train_loss": "", "train_acc": "",
        "val_loss": f"{test_m['loss']:.6f}",
        "val_top1": f"{test_m['top1']:.6f}",
        "val_top5": f"{test_m['top5']:.6f}",
        "val_mae_da": f"{test_m['mae_da']:.1f}",
        "best_epoch": best_epoch,
        "best_val_loss": f"{best_val_loss:.6f}", "lr": "",
        "epoch_time_s": "", "total_time_s": f"{time.time() - training_start:.1f}",
        "gpu_mem_used_mb": gpu_used, "gpu_mem_total_mb": gpu_total,
    })

    return test_m, history

print("Training function defined. Ready to run ablation conditions.")


## 5. Run All Ablation Conditions

This cell iterates through all 7 conditions. If Colab disconnects:
1. Re-run all cells above (they are fast — just data loading)
2. Re-run this cell — completed conditions are skipped, the in-progress condition resumes from its last epoch


In [ ]:
# @title Cell 8: Run all ablation conditions (resumable, OOM-safe)
all_results = {}
all_histories = {}
failed_conditions = []
overall_start = time.time()
condition_times = []

for ci, (name, desc, zero_fields, ctx_override) in enumerate(ABLATION_CONDITIONS):
    cond_start = time.time()
    try:
        test_m, hist = train_ablation(name, desc, zero_fields, ctx_override)
        all_results[name] = test_m
        all_histories[name] = hist
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            print(f"\n  CUDA OOM on {name} — skipping. Error: {e}")
            log_to_csv({
                "timestamp": datetime.now().isoformat(), "condition": name,
                "epoch": 0, "max_epochs": MAX_EPOCHS, "status": "OOM_FAILED",
                "train_loss": "", "train_acc": "", "val_loss": "", "val_top1": "",
                "val_top5": "", "val_mae_da": "", "best_epoch": 0,
                "best_val_loss": "", "lr": "", "epoch_time_s": "",
                "total_time_s": f"{time.time() - cond_start:.1f}",
                "gpu_mem_used_mb": 0, "gpu_mem_total_mb": 0,
            })
            failed_conditions.append((name, str(e)))
            continue
        else:
            raise  # re-raise non-OOM errors

    cond_time = time.time() - cond_start
    condition_times.append(cond_time)
    remaining = len(ABLATION_CONDITIONS) - ci - 1
    if condition_times:
        avg_time = sum(condition_times) / len(condition_times)
        eta_min = remaining * avg_time / 60
        print(f"  ETA: ~{eta_min:.0f} min ({remaining} conditions remaining)")

print(f"\n{'='*70}")
print(f"ALL ABLATION CONDITIONS {'COMPLETE' if not failed_conditions else 'FINISHED (with failures)'}")
print(f"Total time: {(time.time() - overall_start)/60:.1f} min")
print(f"{'='*70}")
if failed_conditions:
    print(f"\nFAILED conditions ({len(failed_conditions)}):")
    for name, err in failed_conditions:
        print(f"  {name}: {err[:80]}")
    print()

print(f"{'Condition':16s} | {'Top-1':>7s} | {'Top-5':>7s} | {'Top-10':>7s} | {'MAE(Da)':>7s} | {'Best Ep':>7s}")
print("-" * 70)
for name, desc, _, _ in ABLATION_CONDITIONS:
    if name in all_results:
        m = all_results[name]
        h = all_histories[name]
        best_ep = max(range(len(h)), key=lambda i: -h[i]["val_loss"]) + 1 if h else 0
        print(f"{name:16s} | {m['top1']:7.4f} | {m['top5']:7.4f} | {m['top10']:7.4f} | {m['mae_da']:7.1f} | {best_ep:7d}")
    else:
        print(f"{name:16s} | {'FAILED':>7s} | {'':>7s} | {'':>7s} | {'':>7s} | {'':>7s}")


## 6. Results & Figures

In [ ]:
# @title Cell 9: Save results JSON
results_summary = {}
for name, desc, zero_fields, ctx_override in ABLATION_CONDITIONS:
    if name not in all_results:
        continue
    m = all_results[name]
    results_summary[name] = {
        "description": desc,
        "zero_fields": zero_fields,
        "context_length": ctx_override if ctx_override is not None else CONTEXT_LENGTH,
        "test_top1": m["top1"],
        "test_top3": m["top3"],
        "test_top5": m["top5"],
        "test_top10": m["top10"],
        "test_mae_da": m["mae_da"],
        "test_loss": m["loss"],
        "n_test": m["n"],
    }

with open(f"{OUTPUT_DIR}/ablation_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print(f"Saved: {OUTPUT_DIR}/ablation_results.json")
print(json.dumps(results_summary, indent=2))


In [ ]:
# @title Cell 10: Ablation figures (3-panel)
import matplotlib.pyplot as plt

# --- Prepare data (skip any failed conditions) ---
conditions = [name for name, _, _, _ in ABLATION_CONDITIONS if name in all_results]
labels = [desc for name, desc, _, _ in ABLATION_CONDITIONS if name in all_results]
metrics = ["top1", "top3", "top5", "top10"]
metric_labels = ["Top-1", "Top-3", "Top-5", "Top-10"]

# Build matrix: rows = conditions, cols = metrics
data = np.array([[all_results[c][m] for m in metrics] for c in conditions])
full_row = data[0]  # Full model is the baseline
delta = (data - full_row) * 100  # percentage points

# Separate feature ablation vs data efficiency
feature_idx = [i for i, c in enumerate(conditions) if c not in DATA_FRACTION and c != "full"]
data_idx = [i for i, c in enumerate(conditions) if c in DATA_FRACTION]

# --- Figure: 3 panels ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), gridspec_kw={"width_ratios": [1.3, 1, 0.8]})

# Panel A: Absolute accuracy heatmap (all conditions)
ax = axes[0]
im = ax.imshow(data * 100, cmap="YlOrRd", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(metric_labels)))
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=10)
for i in range(len(conditions)):
    for j in range(len(metrics)):
        ax.text(j, i, f"{data[i, j]*100:.1f}%", ha="center", va="center",
                fontsize=9, color="white" if data[i, j] > 0.5 else "black")
ax.set_title("A. Absolute Accuracy (%)", fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.8, label="Accuracy (%)")

# Panel B: Feature importance (delta bar chart)
ax = axes[1]
feat_labels = [labels[i] for i in feature_idx]
feat_delta = [delta[i, 0] for i in feature_idx]
colors = ["#d32f2f" if d < -1 else "#f57c00" if d < -0.1 else "#4caf50" for d in feat_delta]
bars = ax.barh(range(len(feat_labels)), feat_delta, color=colors, edgecolor="black", linewidth=0.5)
ax.set_yticks(range(len(feat_labels)))
ax.set_yticklabels(feat_labels, fontsize=11)
ax.set_xlabel("Δ Top-1 Accuracy (pp)", fontsize=11)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("B. Feature Importance", fontsize=13, fontweight="bold")
for i, (bar, val) in enumerate(zip(bars, feat_delta)):
    x_pos = bar.get_width()
    ha = "left" if x_pos >= 0 else "right"
    offset = 0.3 if x_pos >= 0 else -0.3
    ax.text(x_pos + offset, i, f"{val:+.1f}pp", va="center", ha=ha, fontsize=10)

# Panel C: Data efficiency curve
ax = axes[2]
frac_conditions = ["data_25pct", "data_50pct", "data_75pct", "full"]
frac_map = {"data_25pct": 0.25, "data_50pct": 0.50, "data_75pct": 0.75, "full": 1.0}
fracs = [frac_map[c] for c in frac_conditions if c in all_results]
frac_top1 = [all_results[c]["top1"] * 100 for c in frac_conditions if c in all_results]
ax.plot(fracs, frac_top1, "o-", color="#1976d2", linewidth=2, markersize=8)
ax.set_xlabel("Training Data Fraction", fontsize=11)
ax.set_ylabel("Top-1 Accuracy (%)", fontsize=11)
ax.set_title("C. Data Efficiency", fontsize=13, fontweight="bold")
ax.set_xticks(fracs)
ax.set_xticklabels(["25%", "50%", "75%", "100%"])
ax.grid(True, alpha=0.3)
for frac, acc in zip(fracs, frac_top1):
    ax.annotate(f"{acc:.1f}%", (frac, acc), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=10)

plt.tight_layout()
fig_path = f"{OUTPUT_DIR}/ablation_heatmap.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# @title Cell 11: Training curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Validation top-1 accuracy over epochs
ax = axes[0]
for name, desc, _, _ in ABLATION_CONDITIONS:
    if name not in all_histories:
        continue
    hist = all_histories[name]
    epochs = [r["epoch"] for r in hist]
    top1 = [r["val_top1"] for r in hist]
    ax.plot(epochs, [t * 100 for t in top1], label=desc, linewidth=1.5)
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Validation Top-1 Accuracy (%)", fontsize=11)
ax.set_title("A. Training Convergence", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3)

# Panel B: Validation loss over epochs
ax = axes[1]
for name, desc, _, _ in ABLATION_CONDITIONS:
    if name not in all_histories:
        continue
    hist = all_histories[name]
    epochs = [r["epoch"] for r in hist]
    val_loss = [r["val_loss"] for r in hist]
    ax.plot(epochs, val_loss, label=desc, linewidth=1.5)
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Validation Loss", fontsize=11)
ax.set_title("B. Validation Loss", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = f"{OUTPUT_DIR}/ablation_training_curves.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

print("\nDone! All outputs saved to Google Drive.")
